In [1]:
from openai import OpenAI

openai_client = OpenAI()

In [2]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

In [3]:
from minsearch import AppendableIndex

In [4]:
index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [5]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

In [6]:
import json

RAG_INSTRUCTIONS = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

RAG_PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return RAG_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )

In [7]:
question = "How do I create a dahsbord in Evidently?"
search_results = search(question)
user_prompt = build_prompt(question, search_results)

In [8]:
messages = [
    {"role": "system", "content": RAG_INSTRUCTIONS},
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
)

In [9]:
print(response.output_text)

The provided context does not include specific instructions on how to create a dashboard in Evidently.


In [10]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Use only facts from the knowledge base when answering.
IMPORTANT: f you cannot find the answer, inform the user.
"""

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}

In [11]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
)

response.usage.input_tokens

63

In [12]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
)

response.usage.input_tokens

110

In [13]:
tool_call = response.output[0]
tool_call

ResponseFunctionToolCall(arguments='{"query":"create a dashboard in Evidently"}', call_id='call_TlPgyLL4EWpeqJyTNTcoFZGG', name='search', type='function_call', id='fc_0e62f2c650882584006acb7838c8b087d1a0112d31dabae10b', async_=None, caller=None, namespace=None, status='completed')

In [14]:
messages.append(tool_call)

In [15]:
tool_call.arguments

'{"query":"create a dashboard in Evidently"}'

In [16]:
arguments = json.loads(tool_call.arguments)
arguments

{'query': 'create a dashboard in Evidently'}

In [17]:
search_results = search(query='create dashboard in Evidently')

In [18]:
search_results = search(**arguments)
search_results[:1]

[{'start': 0,
  'content': "<Check>\n  Dashboard is available in **Evidently OSS**, **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\n## What is a Dashboard?\n\nA Dashboard provides a clear view of your AI application performance. You can use it:\n\n- to track evaluation results across multiple experiments;\n- to track live production quality over time.\n\nEach Project has its own Dashboard. It's empty at first.\n\nTo populate it, you need to run an evaluation and **save at least one Report** to the Project. You can then choose values from Reports to plot.\n\n![](/images/dashboard_llm_light.png)\n\n## Dashboard Tabs\n\n<Check>\n  Multiple Tabs are available in **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\nYou can logically organize Panels within the same Dashboard into different Tabs.\n\n![](/images/dashboard_llm_tabs.gif)\n\n## **Dashboard Panels**\n\nA Panel is a visual element in the Dashboard that displays specified values in a single widget. Panels ca

In [19]:
call_output = {
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": json.dumps(search_results),
}

In [20]:
messages.append(call_output)
messages

[{'role': 'system',
  'content': "\nYou're a documentation assistant. \n\nAnswer the user question using the documentation knowledge base\n\nUse only facts from the knowledge base when answering.\nIMPORTANT: f you cannot find the answer, inform the user.\n"},
 {'role': 'user', 'content': 'How do I create a dahsbord in Evidently?'},
 ResponseFunctionToolCall(arguments='{"query":"create a dashboard in Evidently"}', call_id='call_TlPgyLL4EWpeqJyTNTcoFZGG', name='search', type='function_call', id='fc_0e62f2c650882584006acb7838c8b087d1a0112d31dabae10b', async_=None, caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_TlPgyLL4EWpeqJyTNTcoFZGG',
  'output': '[{"start": 0, "content": "<Check>\\n  Dashboard is available in **Evidently OSS**, **Evidently Cloud** and **Evidently Enterprise**.\\n</Check>\\n\\n## What is a Dashboard?\\n\\nA Dashboard provides a clear view of your AI application performance. You can use it:\\n\\n- to track evaluatio

In [21]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
)

response.usage.input_tokens

3686

In [22]:
from typing import Literal
from pydantic import BaseModel, Field


class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [23]:
response = openai_client.responses.parse(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
    text_format=RAGResponse
)

response.usage.input_tokens

3910

In [24]:
rag_response = response.output_parsed
print(rag_response.answer)

To create a dashboard in Evidently, follow these steps:

### 1. **Create a Project**
   Ensure that you have created a Project in Evidently. Dashboards are tied to Projects, so you need at least one Project created.

### 2. **Run an Evaluation**
   Perform and save at least one Report in your Project. The dashboard will require Reports as a data source to display metrics.

### 3. **Access the Dashboard**
   Once you save a Report, you can access the Dashboard within your Project. Initially, it will be empty.

### 4. **Add Panels**
   You can add visual elements called **Panels** to your Dashboard. Each Panel visualizes a specific value or metric. You can do this in two ways:
   - **Using the Python API**:
     You can define your Dashboard and add Panels programmatically. Here’s a basic example to add a text panel:
     ```python
     from evidently.sdk.panels import DashboardPanelPlot

     project.dashboard.add_panel(
         DashboardPanelPlot(
             title="Dashboard Title",

In [25]:
##Agentic Loop

In [26]:
def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'search':
        result = search(**arguments)
    else: 
        result = 'not found tool "{name}"'

    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

In [27]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

IMPORTANT: When you explore the knowledge base, make at least 3 different
searchers to make sure you explore the topic well.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

In [28]:
question = "How do I create a dahsbord in Evidently?"

In [29]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

In [30]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=message_history,
    tools=[search_tool],
)

In [31]:
message_history.extend(response.output)

In [32]:
message_history

[{'role': 'system',
  'content': "\nYou're a documentation assistant. \n\nAnswer the user question using the documentation knowledge base\n\nIMPORTANT: When you explore the knowledge base, make at least 3 different\nsearchers to make sure you explore the topic well.\n\nUse only facts from the knowledge base when answering.\nIf you cannot find the answer, inform the user.\n\nOur knowledge base is entirely about Evidently, so you don't need to \ninclude the word 'evidently' in search results\n"},
 {'role': 'user', 'content': 'How do I create a dahsbord in Evidently?'},
 ResponseFunctionToolCall(arguments='{"query":"create dashboard"}', call_id='call_1lpIR0oJOLOZ2jU8M9ULEKuY', name='search', type='function_call', id='fc_087b2bff6f41a09a006acb7844e1f487d1876f8098819f4fdd', async_=None, caller=None, namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"dashboard setup"}', call_id='call_2qWtv46AZz86odw1AooeVxC0', name='search', type='function_call', id='fc_087b2

In [33]:
for message in response.output:
    if message.type == 'function_call':
        print(f'executing {message.name}({message.arguments})...')
        tool_call_output = make_call(message)
        message_history.append(tool_call_output)

executing search({"query":"create dashboard"})...
executing search({"query":"dashboard setup"})...
executing search({"query":"dashboard design"})...


In [34]:
len(message_history)

8

In [35]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=message_history,
    tools=[search_tool],
)

response.usage.input_tokens

12147

In [36]:
print(response.output_text)

To create a dashboard in Evidently, you can use either the user interface or the Python API. Here’s a step-by-step guide for both methods:

### Using the User Interface

1. **Enter Edit Mode:**
   - Open your dashboard and click on the edit icon in the top right corner.

2. **Add Tabs:**
   - Click the plus sign to add a new tab. You can create a custom tab by selecting "empty" and naming it.

3. **Add a Panel:**
   - Click on the "Add Panel" button.
   - Follow the prompts to configure the panel:
     - Select the metric(s) to visualize.
     - Optionally apply filters by tags.
     - Use the preview to review your panel setup.
   - Click "Save" and choose the tab where you want to place the panel.

4. **Repeat for More Panels:**
   - You can repeat the steps to add multiple panels, including text panels, counters, graphs, etc.

### Using the Python API

1. **Connect to Evidently Cloud:**
   - Ensure you are connected to the Evidently Cloud and have created a project.

2. **Add a New 

In [37]:
## Agentic Loop

In [38]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search
2) in the second interation, analyze the results from the previous search
   and perform 2 more searches
3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

In [39]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=[search_tool],
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()

    if not has_function_calls:
        break

iteraration number 1...
ASSISTANT: Creating a dashboard is an essential task for visualizing and monitoring your data effectively. I will first search for how to create a dashboard, which should provide the foundational steps and any necessary tools or components involved in this process. 

Let's start by finding documentation specific to creating dashboards.
executing search({"query":"create dashboard"})...

iteraration number 2...
ASSISTANT: The initial search returned useful information on creating a dashboard, emphasizing the importance of adding panels and organizing them into tabs. To create a dashboard, you can use Python API methods for adding panels and configuring tabs. Additionally, it's important to note that you must have at least one report saved to your project before you can populate the dashboard with data.

Now, I will search for more detailed instructions on how to add and manage panels, specifically through the user interface, as this could provide different insight

In [40]:
#adding more toolS

In [41]:
def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

add_entry_tool = {
    "type": "function",
    "name": "add_entry",
    "description": "Add a new documentation entry to the index.",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "The source filename associated with the entry"
            },
            "title": {
                "type": "string",
                "description": "The title of the documentation entry"
            },
            "description": {
                "type": "string",
                "description": "A short description summarizing the entry"
            },
            "content": {
                "type": "string",
                "description": "The full content of the documentation entry"
            }
        },
        "required": [
            "filename",
            "title",
            "description",
            "content"
        ]
    }
}

In [42]:
def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'search':
        result = search(**arguments)
    elif name == 'add_entry':
        result = add_entry(**arguments)
    else: 
        result = 'not found tool "{name}"'

    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

In [43]:
#make tools work together

In [44]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]
tools = [search_tool, add_entry_tool]

iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=tools,
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()

    if not has_function_calls:
        break

iteraration number 1...
ASSISTANT: To provide you with accurate information on creating a dashboard, I will first search for general documentation related to dashboard creation. This will help me understand the foundational steps and components involved in the process.

Let's perform the first search now.
executing search({"query":"create dashboard"})...

iteraration number 2...
ASSISTANT: The first search returned helpful information about creating a dashboard, specifically around adding panels and organizing tabs. It indicates that dashboards are an interface for visualizing AI application performance, and you can populate them with data from evaluation reports. 

To further understand the specific steps needed to create a dashboard from start to finish, I will conduct additional searches focusing on how to set up an initial dashboard and details on the user interface for this process.

Let's proceed with the next two searches. The first will focus on overall dashboard setup, and the

In [45]:
message_history.append(
    {"role": "user", "content": "add this content to our database"}
)

In [46]:
while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=tools,
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()

    if not has_function_calls:
        break

iteraration number 4...
executing add_entry({"filename":"docs/platform/dashboard_creation.mdx","title":"Creating a Dashboard","description":"Comprehensive guide to creating and managing a dashboard.","content":"### Steps to Create a Dashboard\n\n1. **Initial Setup**:\n   - Ensure you're connected to the cloud service and have created a project where your dashboard will reside.\n\n2. **Adding Tabs**:\n   - To organize your dashboard, you can add multiple tabs. Here’s how:\n     - Enter \"Edit\" mode on the dashboard.\n     - Click the plus sign labeled \"add Tab\" on the left.\n     - You can create a new custom tab or use pre-built templates that organize related metrics.\n\n3. **Adding Panels**:\n   - Panels are the core visual components of your dashboard. Here’s how to add them:\n     - Once you're in \"Edit\" mode, click on the “Add Panel” button.\n     - Follow the prompts to configure the panel. You can choose panel types like counters, pie charts, or line plots.\n     - Use the 

In [47]:
index.docs[-1]

{'start': 0,
 'content': '### Steps to Create a Dashboard\n\n1. **Initial Setup**:\n   - Ensure you\'re connected to the cloud service and have created a project where your dashboard will reside.\n\n2. **Adding Tabs**:\n   - To organize your dashboard, you can add multiple tabs. Here’s how:\n     - Enter "Edit" mode on the dashboard.\n     - Click the plus sign labeled "add Tab" on the left.\n     - You can create a new custom tab or use pre-built templates that organize related metrics.\n\n3. **Adding Panels**:\n   - Panels are the core visual components of your dashboard. Here’s how to add them:\n     - Once you\'re in "Edit" mode, click on the “Add Panel” button.\n     - Follow the prompts to configure the panel. You can choose panel types like counters, pie charts, or line plots.\n     - Use the preview option to adjust your settings, then click "Save" once you’re satisfied.\n\n4. **Configuring Panels**:\n   - When adding a panel, specify:\n     - **Title**: Name for the panel.\n  

In [48]:
#The QA Loop

In [49]:
tools = [search_tool, add_entry_tool]

message_history = [
    {"role": "system", "content": instructions},
]

iteration_number = 1

# Q&A loop
while True:
    user_prompt = input('You:')
    if user_prompt.lower().strip() == 'stop':
        break

    message_history.append({"role": "user", "content": user_prompt})

    # tool-call loop
    while True:
        response = openai_client.responses.create(
            model='gpt-4o-mini',
            input=message_history,
            tools=tools,
        )

        print(f'iteraration number {iteration_number}...') 
        message_history.extend(response.output)

        has_function_calls = False

        for message in response.output:
            if message.type == 'function_call':
                print(f'executing {message.name}({message.arguments})...')
                tool_call_output = make_call(message)
                message_history.append(tool_call_output)
                has_function_calls = True

            if message.type == 'message':
                text = message.content[0].text
                print('ASSISTANT:', text)

        iteration_number = iteration_number + 1
        print()

        if not has_function_calls:
            break

You: stop


In [50]:
class Agent:

    def __init__(self, llm_client, model, instructions, tools):
        self.llm_client = llm_client
        self.model = model
        self.instructions = instructions
        self.tools = tools

    def make_call(self, tool_call):
        arguments = json.loads(tool_call.arguments)
        name = tool_call.name

        if name == 'search':
            result = search(**arguments)
        elif name == 'add_entry':
            result = add_entry(**arguments)
        else:
            result = 'not found tool "{name}"'

        return {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": json.dumps(result),
        }   

    def loop(self, user_prompt, message_history=None):
        if not message_history:
            message_history = [
                {"role": "system", "content": self.instructions},
            ]

        message_history.append({"role": "user", "content": user_prompt})

        iteration_number = 0

        while True:
            response = self.llm_client.responses.create(
                model=self.model,
                input=message_history,
                tools=self.tools,
            )

            print(f'iteraration number {iteration_number}...') 
            message_history.extend(response.output)

            has_function_calls = False

            for message in response.output:
                if message.type == 'function_call':
                    print(f'executing {message.name}({message.arguments})...')
                    tool_call_output = self.make_call(message)
                    message_history.append(tool_call_output)
                    has_function_calls = True

                if message.type == 'message':
                    text = message.content[0].text
                    print('ASSISTANT:', text)

            iteration_number = iteration_number + 1
            print()

            if not has_function_calls:
                break

        return message_history        

    def qna(self):
        message_history = [
            {"role": "system", "content": instructions},
        ]

        iteration_number = 1

        # Q&A loop
        while True:
            user_prompt = input('You:')
            if user_prompt.lower().strip() == 'stop':
                break

            message_history = self.loop(user_prompt, message_history)

In [51]:
agent = Agent(
    llm_client=OpenAI(),
    model='gpt-4o-mini',
    instructions=instructions,
    tools=tools
)

In [52]:
messages = agent.loop('evidently adshboards')

iteraration number 0...
ASSISTANT: To begin addressing your query about "dashboards," I'll perform a search using this keyword. Dashboards often play a critical role in data visualization and monitoring, and understanding how they are implemented or can be utilized within the platform will be essential for providing you with relevant information.

### Performing the search...
executing search({"query":"dashboards"})...

iteraration number 1...
ASSISTANT: The search for "dashboards" yielded several results related to how dashboards can be created, customized, and utilized within the platform. Key points included methods for adding tabs and panels both through the user interface and via a Python API. It also mentioned how to manage configurations and the types of panels available.

### Analyzing Results
The results highlighted the importance of dashboards in visualizing data and monitoring key metrics. I want to perform additional searches focusing specifically on the different types of 

In [53]:
agent.qna()

You: stop


In [54]:
# toykit as a stepping stone

In [55]:
!uv add toyaikit

Resolved 142 packages in 52ms
Checked 138 packages in 370ms


In [56]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [57]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": ["query"]
    }
}

In [58]:
def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

add_entry_tool = {
    "type": "function",
    "name": "add_entry",
    "description": "Add a new documentation entry to the index.",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "The source filename associated with the entry"
            },
            "title": {
                "type": "string",
                "description": "The title of the documentation entry"
            },
            "description": {
                "type": "string",
                "description": "A short description summarizing the entry"
            },
            "content": {
                "type": "string",
                "description": "The full content of the documentation entry"
            }
        },
        "required": ["filename", "title", "description", "content"]
    }
}

In [59]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search

2) in the second interation, analyze the results from the previous search
   and perform 2 more searches

3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

from toyaikit.tools import Tools

from toyaikit.tools import Tools

In [60]:
from toyaikit.tools import Tools

In [59]:
# creating agent

In [61]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)
agent_tools.add_tool(add_entry, add_entry_tool)

agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the documentation database for relevant results based on a query string.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'The search query to look up in the index'}},
   'required': ['query']}},
 {'type': 'function',
  'name': 'add_entry',
  'description': 'Add a new documentation entry to the index.',
  'parameters': {'type': 'object',
   'properties': {'filename': {'type': 'string',
     'description': 'The source filename associated with the entry'},
    'title': {'type': 'string',
     'description': 'The title of the documentation entry'},
    'description': {'type': 'string',
     'description': 'A short description summarizing the entry'},
    'content': {'type': 'string',
     'description': 'The full content of the documentation entry'}},
   'required': ['filename', 'title', 'description', 'content']}}]

In [63]:
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.llm import OpenAIClient
from openai import OpenAI

llm_client = OpenAIClient(
    model="gpt-4o-mini",
    client=OpenAI()
)

In [64]:
agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    # chat_interface=,
    llm_client=llm_client
)

In [65]:
# Running the Agent

In [66]:
result = agent.loop('how can I create evidently dahsbord?')

In [67]:
print(result.last_message)

To create a dashboard, you need to follow these key steps:

1. **Set Up Your Project**: Ensure you are connected to the platform and have created a project. A dashboard is associated with a project which contains reports of evaluation results.

2. **Add Tabs**: You can add multiple tabs to organize your dashboard. Enter "Edit" mode, click on the "+" sign to add a new tab, or directly create a tab while adding a panel if it does not already exist.

3. **Adding Panels**: You can add various types of panels to your dashboard, including counters, line plots, bar plots, and text panels. 

   - **Using the Python API**:
     - First, import the required libraries:
       ```python
       from evidently.sdk.models import PanelMetric
       from evidently.sdk.panels import DashboardPanelPlot
       ```
     - To add a panel, you can use code like the following:
       ```python
       project.dashboard.add_panel(
           DashboardPanelPlot(
               title="Panel Title",
              

In [68]:
result.cost.total_cost

Decimal('0.0029625')

In [69]:
# adding visability

In [70]:
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback

In [71]:
chat_interface = IPythonChatInterface()
runner_callback = DisplayingRunnerCallback(chat_interface=chat_interface)

In [72]:
result = agent.loop(
    'how can I create evidently dahsbard?',
    callback=runner_callback
)

-> Response received


-> Response received


-> Response received


In [73]:
#continuing the conversation

In [74]:
result2 = agent.loop(
    'show me the code',
    callback=runner_callback,
    previous_messages=result.all_messages
)

-> Response received


In [75]:
# Q and A Loop

In [76]:
agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [77]:
result = agent.run()

You: stop


Chat ended.


In [78]:
result.cost.total_cost

Decimal('0.000')

In [85]:
#organizing tools in a class

In [79]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [80]:
def search(query: str) -> list[dict]:
    """
    Search the documentation database for relevant results based on a query string.

    Args:
        query (str): The search query to look up in the index.

    Returns:
        list[dict]: A list of search result objects returned by the index.
    """
    results = index.search(
        query=query,
        num_results=5
    )
    return results

def add_entry(filename, title, description, content):
    """
    Add a new documentation entry to the index.

    Args:
        filename (str): The source filename associated with the entry.
        title (str): The title of the documentation entry.
        description (str): A short description summarizing the entry.
        content (str): The full content of the documentation entry.

    Returns:
        str: A status message indicating the result of the operation ("OK").
    """
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

In [81]:
from toyaikit.tools import Tools

agent_tools = Tools()
agent_tools.add_tool(search)
agent_tools.add_tool(add_entry)

agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the documentation database for relevant results based on a query string.\n\nArgs:\n    query (str): The search query to look up in the index.\n\nReturns:\n    list[dict]: A list of search result objects returned by the index.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}},
 {'type': 'function',
  'name': 'add_entry',
  'description': 'Add a new documentation entry to the index.\n\nArgs:\n    filename (str): The source filename associated with the entry.\n    title (str): The title of the documentation entry.\n    description (str): A short description summarizing the entry.\n    content (str): The full content of the documentation entry.\n\nReturns:\n    str: A status message indicating the result of the operation ("OK").',
  'parameters': {'type': 'object',
   'properties': {'filenam

In [82]:
from openai import OpenAI
from toyaikit.llm import OpenAIClient
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback

llm_client = OpenAIClient(
    model="gpt-4o-mini",
    client=OpenAI()
)

chat_interface = IPythonChatInterface()
runner_callback = DisplayingRunnerCallback(chat_interface=chat_interface)

In [83]:
agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [84]:
result = agent.loop(
    'how can I create evidently dahsbard?',
    callback=runner_callback
)

-> Response received


-> Response received


-> Response received
